# 2.1 Breadth first and uniform cost search: practice notebook

Breadth first search explores level by level and finds the fewest actions. Uniform cost search explores in order of cost and finds the cheapest path. Learn both, their costs, and the three lemma proof that UCS is optimal.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Breadth first search on the water jugs

In [ ]:
# Breadth first search on the water jug problem (jugs of 3 and 4 gallons).
# Goal: either jug holds exactly 2 gallons. The frontier is a plain list used as a queue.
def successors(state):
    a, b = state
    pour_ab = min(a, 4 - b)
    pour_ba = min(b, 3 - a)
    return [(3, b), (a, 4), (0, b), (a, 0), (a - pour_ab, b + pour_ab), (a + pour_ba, b - pour_ba)]

def bfs(start):
    frontier = [[start]]          # a list of paths, oldest first
    seen = [start]
    while len(frontier) > 0:
        path = frontier.pop(0)    # take the OLDEST path (first in, first out)
        for child in successors(path[-1]):
            if child not in seen:
                new_path = path + [child]
                if 2 in child:    # goal test when the child is generated
                    return new_path
                seen.append(child)
                frontier.append(new_path)
    return None

answer = bfs((0, 0))
print("path:", answer)
print("actions needed:", len(answer) - 1)

### Uniform cost search from Sibiu to Bucharest

In [ ]:
# Uniform cost search on the slide 34 problem: from Sibiu to Bucharest.
# The frontier holds (cost, path) pairs. We always take out the cheapest one.
roads = {
    "Sibiu": [("Rimnicu Vilcea", 80), ("Fagaras", 99)],
    "Rimnicu Vilcea": [("Pitesti", 97)],
    "Fagaras": [("Bucharest", 211)],
    "Pitesti": [("Bucharest", 101)],
    "Bucharest": [],
}

def ucs(start, goal):
    frontier = [(0, [start])]
    done = []
    while len(frontier) > 0:
        frontier.sort()                 # cheapest path first
        cost, path = frontier.pop(0)
        city = path[-1]
        print("take out", city, "with cost", cost)
        if city == goal:                # goal test when the path is taken OUT
            return cost, path
        done.append(city)
        for nxt, step in roads[city]:
            if nxt not in done:
                frontier.append((cost + step, path + [nxt]))
    return None

print(ucs("Sibiu", "Bucharest"))

## Your turn

### Task 1: Shortest path with breadth first search

Write `bfs_path(graph, start, goal)`. graph is a dict from a node to the list of its neighbours. Return the list of nodes on a path with the fewest edges, or None if there is none. Visit neighbours in the order of the list.

Hint: Keep a list of paths. Take the oldest out with pop(0) and add each new path at the end.

In [ ]:
def bfs_path(graph, start, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("bfs_path({'A': ['B', 'C'], 'B': ['D'], 'C': ['D', 'E'], 'D': ['F'], 'E': ['F'], 'F': []}, 'A', 'F')", ['A', 'B', 'D', 'F']), ("bfs_path({'A': ['B'], 'B': ['A'], 'C': []}, 'A', 'C')", None), ("bfs_path({'A': ['B'], 'B': []}, 'A', 'A')", ['A']), ("bfs_path({'S': ['A', 'G'], 'A': ['G'], 'G': []}, 'S', 'G')", ['S', 'G'])])

### Task 2: Cheapest cost with uniform cost search

Write `ucs_cost(edges, start, goal)`. edges is a list of [a, b, cost] roads that can be driven both ways. Return the lowest total cost from start to goal, or -1 if the goal cannot be reached.

Hint: Keep (cost, node) pairs, sort, take the first out. Skip a node you have already taken out. Test the goal when you take it out.

In [ ]:
def ucs_cost(edges, start, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("ucs_cost([['S', 'A', 1], ['S', 'B', 4], ['A', 'B', 2], ['A', 'C', 5], ['B', 'C', 1], ['C', 'G', 3]], 'S', 'G')", 7), ("ucs_cost([['Sibiu', 'RV', 80], ['Sibiu', 'Fagaras', 99], ['RV', 'Pitesti', 97], ['Fagaras', 'Bucharest', 211], ['Pitesti', 'Bucharest', 101]], 'Sibiu', 'Bucharest')", 278), ("ucs_cost([['A', 'B', 2]], 'A', 'C')", -1), ("ucs_cost([], 'A', 'A')", 0)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Shortest path with breadth first search

The queue makes every path of length k come out before any path of length k + 1.

In [ ]:
def bfs_path(graph, start, goal):
    if start == goal:
        return [start]
    frontier = [[start]]
    seen = [start]
    while len(frontier) > 0:
        path = frontier.pop(0)
        for nxt in graph[path[-1]]:
            if nxt not in seen:
                if nxt == goal:
                    return path + [nxt]
                seen.append(nxt)
                frontier.append(path + [nxt])
    return None

In [ ]:
check([("bfs_path({'A': ['B', 'C'], 'B': ['D'], 'C': ['D', 'E'], 'D': ['F'], 'E': ['F'], 'F': []}, 'A', 'F')", ['A', 'B', 'D', 'F']), ("bfs_path({'A': ['B'], 'B': ['A'], 'C': []}, 'A', 'C')", None), ("bfs_path({'A': ['B'], 'B': []}, 'A', 'A')", ['A']), ("bfs_path({'S': ['A', 'G'], 'A': ['G'], 'G': []}, 'S', 'G')", ['S', 'G'])])

### Solution to task 2: Cheapest cost with uniform cost search

S, A, B, C, G costs 1 + 2 + 1 + 3 = 7, which beats S, B, C, G = 8.

In [ ]:
def ucs_cost(edges, start, goal):
    frontier = [(0, start)]
    done = []
    while len(frontier) > 0:
        frontier.sort()
        cost, node = frontier.pop(0)
        if node in done:
            continue
        if node == goal:
            return cost
        done.append(node)
        for a, b, c in edges:
            if a == node and b not in done:
                frontier.append((cost + c, b))
            if b == node and a not in done:
                frontier.append((cost + c, a))
    return -1

In [ ]:
check([("ucs_cost([['S', 'A', 1], ['S', 'B', 4], ['A', 'B', 2], ['A', 'C', 5], ['B', 'C', 1], ['C', 'G', 3]], 'S', 'G')", 7), ("ucs_cost([['Sibiu', 'RV', 80], ['Sibiu', 'Fagaras', 99], ['RV', 'Pitesti', 97], ['Fagaras', 'Bucharest', 211], ['Pitesti', 'Bucharest', 101]], 'Sibiu', 'Bucharest')", 278), ("ucs_cost([['A', 'B', 2]], 'A', 'C')", -1), ("ucs_cost([], 'A', 'A')", 0)])